# Lecture 3 — Linear Classification: Hinge Loss, Regularization, and Optimization

Learning-first implementation: NumPy from scratch first, then a small scikit-learn comparison.

$$f(x)=\theta^T x+\theta_0$$

$$J(\theta,\theta_0;\alpha)=L(\theta,\theta_0)+\alpha R(\theta).$$


## 1. Dataset

A small binary classification dataset with labels $+1$ and $-1$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = np.array([[4,4],[5,3],[3,5],[1,1],[2,1],[1,2],[4,1],[2,5],[2,2],[1,5],[1,3],[3,4],[5,4]], dtype=float)
y = np.array([1,1,1,-1,-1,-1,-1,1,-1,1,-1,1,1], dtype=float)

print('X shape:', X.shape)
print('First example:', X[0], 'label:', y[0])

plt.figure(figsize=(7,6))
plt.scatter(X[y==1,0], X[y==1,1], marker='o', s=90, label='+1')
plt.scatter(X[y==-1,0], X[y==-1,1], marker='x', s=90, label='-1')
plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Lecture 3 training data'); plt.grid(True); plt.legend(); plt.show()


## 2. Score, margin, and hinge loss

$$z_i=y_i(\theta^T x_i+\theta_0).$$

$$L_i=\max(0,1-z_i).$$

If $z_i\ge1$, the desired margin is satisfied and the hinge loss is zero.


In [ ]:
def score(x, theta, theta0):
    return np.dot(theta, x) + theta0

def prediction(x, theta, theta0):
    return 1 if score(x, theta, theta0) >= 0 else -1

def margin(x, yi, theta, theta0):
    return yi * score(x, theta, theta0)

def hinge_loss(z):
    return max(0.0, 1.0 - z)

example_theta = np.array([1., 2.])
example_bias = -5.
print('Example score:', score(X[0], example_theta, example_bias))
print('Example prediction:', prediction(X[0], example_theta, example_bias))
print('Example margin:', margin(X[0], y[0], example_theta, example_bias))

z = np.linspace(-2, 3, 300)
plt.figure(figsize=(7,5))
plt.plot(z, [hinge_loss(v) for v in z], linewidth=2)
plt.axvline(1, linestyle='--', label='margin = 1')
plt.xlabel('signed margin z'); plt.ylabel('hinge loss'); plt.title('Hinge loss'); plt.grid(True); plt.legend(); plt.show()


## 3. Complete objective

$$J(\theta,\theta_0;\alpha)=\frac{1}{n}\sum_{i=1}^{n}\max\left(0,1-y_i(\theta^T x_i+\theta_0)\right)+\frac{\alpha}{2}\|\theta\|_2^2.$$

We do not regularize the bias $\theta_0$.


In [ ]:
def average_hinge_loss(X, y, theta, theta0):
    margins = y * (X @ theta + theta0)
    return np.mean(np.maximum(0.0, 1.0 - margins))

def regularization(theta):
    return 0.5 * np.dot(theta, theta)

def objective(X, y, theta, theta0, alpha):
    return average_hinge_loss(X, y, theta, theta0) + alpha * regularization(theta)

def training_error(X, y, theta, theta0):
    predictions = np.where(X @ theta + theta0 >= 0, 1, -1)
    return np.mean(predictions != y)

alpha = 0.1
print('Average hinge loss:', average_hinge_loss(X, y, example_theta, example_bias))
print('Regularization:', regularization(example_theta))
print('Objective J:', objective(X, y, example_theta, example_bias, alpha))
print('Training error:', training_error(X, y, example_theta, example_bias))


## 4. Gradient descent

For active examples ($z_i<1$):

$$\nabla_\theta J=-\frac{1}{n}\sum_{i:z_i<1}y_i x_i+\alpha\theta$$

$$\frac{\partial J}{\partial\theta_0}=-\frac{1}{n}\sum_{i:z_i<1}y_i.$$

Updates:

$$\theta\leftarrow\theta-\eta\nabla_\theta J$$

$$\theta_0\leftarrow\theta_0-\eta\frac{\partial J}{\partial\theta_0}.$$


In [ ]:
def gradient(X, y, theta, theta0, alpha):
    margins = y * (X @ theta + theta0)
    active = margins < 1.0
    grad_theta = -(X[active].T @ y[active]) / len(X)
    grad_theta0 = -np.sum(y[active]) / len(X)
    grad_theta += alpha * theta
    return grad_theta, grad_theta0

def optimize(X, y, theta, theta0, alpha=0.1, learning_rate=0.01, epochs=1000):
    theta = np.asarray(theta, dtype=float).copy()
    theta0 = float(theta0)
    history = {'objective': [], 'hinge': [], 'reg': [], 'error': [], 'theta1': [], 'theta2': [], 'theta0': []}
    for epoch in range(epochs):
        history['objective'].append(objective(X, y, theta, theta0, alpha))
        history['hinge'].append(average_hinge_loss(X, y, theta, theta0))
        history['reg'].append(alpha * regularization(theta))
        history['error'].append(training_error(X, y, theta, theta0))
        history['theta1'].append(theta[0]); history['theta2'].append(theta[1]); history['theta0'].append(theta0)
        grad_theta, grad_theta0 = gradient(X, y, theta, theta0, alpha)
        theta -= learning_rate * grad_theta
        theta0 -= learning_rate * grad_theta0
        if epoch % 100 == 0 or epoch == epochs - 1:
            print(f'Epoch {epoch:4d} | J={history["objective"][-1]:.6f} | error={history["error"][-1]:.4f} | theta={theta} | theta0={theta0:.4f}')
    return theta, theta0, history

initial_theta = np.array([1., 2.])
initial_bias = -5.
final_theta, final_bias, history = optimize(X, y, initial_theta, initial_bias)
print('Final parameters:', final_theta, final_bias)


## 5. Optimization plots


In [ ]:
def plot_boundary(theta, theta0, label):
    xs = np.linspace(0, 6, 200)
    if abs(theta[1]) > 1e-12:
        ys = -(theta[0] * xs + theta0) / theta[1]
        plt.plot(xs, ys, linewidth=2, label=label)

plt.figure(figsize=(7,6))
plt.scatter(X[y==1,0], X[y==1,1], marker='o', s=90, label='+1')
plt.scatter(X[y==-1,0], X[y==-1,1], marker='x', s=90, label='-1')
plot_boundary(initial_theta, initial_bias, 'initial boundary')
plot_boundary(final_theta, final_bias, 'optimized boundary')
plt.xlim(0,6); plt.ylim(0,6); plt.xlabel('x1'); plt.ylabel('x2'); plt.title('Decision boundary'); plt.grid(True); plt.legend(); plt.show()

plt.figure(figsize=(7,5)); plt.plot(history['objective'], linewidth=2); plt.xlabel('iteration'); plt.ylabel('J'); plt.title('Objective'); plt.grid(True); plt.show()
plt.figure(figsize=(7,5)); plt.plot(history['hinge'], label='hinge loss', linewidth=2); plt.plot(history['reg'], label='alpha * regularization', linewidth=2); plt.xlabel('iteration'); plt.ylabel('value'); plt.title('Objective components'); plt.grid(True); plt.legend(); plt.show()
plt.figure(figsize=(7,5)); plt.plot(history['error'], linewidth=2); plt.xlabel('iteration'); plt.ylabel('training error'); plt.title('Training error'); plt.grid(True); plt.show()
plt.figure(figsize=(7,6)); plt.plot(history['theta1'], history['theta2'], linewidth=2); plt.scatter(history['theta1'][0], history['theta2'][0], s=90, label='start'); plt.scatter(history['theta1'][-1], history['theta2'][-1], s=90, marker='x', label='end'); plt.xlabel('theta1'); plt.ylabel('theta2'); plt.title('Parameter path'); plt.grid(True); plt.legend(); plt.show()


## 6. Experiment with alpha

For each fixed $\alpha$, the optimizer solves:

$$\left(\theta^*(\alpha),\theta_0^*(\alpha)\right)=\arg\min_{\theta,\theta_0}J(\theta,\theta_0;\alpha).$$


In [ ]:
alphas = [0.0, 0.01, 0.1, 0.5, 1.0]
results = []
for a in alphas:
    t, b, _ = optimize(X, y, np.zeros(2), 0.0, alpha=a, learning_rate=0.01, epochs=500)
    results.append((a, objective(X, y, t, b, a), training_error(X, y, t, b), np.linalg.norm(t)))

print('alpha | objective | training_error | ||theta||')
for a, obj, err, norm in results:
    print(f'{a:5.2f} | {obj:9.5f} | {err:14.4f} | {norm:8.4f}')


## 7. External library comparison

Only now do we compare with scikit-learn. This does not replace our from-scratch implementation.


In [ ]:
from sklearn.linear_model import SGDClassifier

sk_model = SGDClassifier(loss='hinge', penalty='l2', alpha=0.1, max_iter=2000, tol=1e-6, random_state=42)
sk_model.fit(X, y)
print('scikit-learn training accuracy:', sk_model.score(X, y))
print('scikit-learn theta:', sk_model.coef_[0])
print('scikit-learn bias:', sk_model.intercept_[0])


## 8. Lecture 3 → Lecture 4

Lecture 3 solves the **inner optimization** for a fixed $\alpha$.

Lecture 4 adds the **outer model-selection problem**: use cross-validation to choose the $\alpha$ that generalizes best.

$$\alpha^*=\arg\max_{\alpha}S(\alpha).$$
